# M03A — Bayesian Decision & Decision Boundaries

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tulip-lab/pattern-classification-lab/blob/develop/M03-Decision-Theory/M03A-Bayesian-Decision-and-Boundaries.ipynb)

**Pattern Classification practical · approximately 2 hours · M03 · CLO1, CLO3**

This is the canonical student-facing practical for session `M03A`.
It was newly authored for the Pattern Classification Lab using concepts and
public migration references from `tulip-lab/flip01`: `F1A-01-DecisionBoundaries.ipynb`. No legacy
notebook cells or outputs are copied verbatim. The implementation uses current
public APIs and synthetic or scikit-learn-bundled data.

**Licence:** code cells are available under MPL-2.0; original narrative teaching
content is available under CC BY-NC-SA 4.0. Third-party works and datasets retain
their own terms. See the repository `LICENSE`, `CONTENT-LICENSE.md`, and
`NOTICE.md`.

## Overview

Bayesian classification turns probability models and a decision rule into an explicit boundary. You will estimate two Gaussian class models, compute posterior probabilities, and observe how prior beliefs change predictions.

Work through the guided cells first, then complete the challenge in your own
copy. Keep your interpretation beside the code: a result without an argument
is not complete evidence.

## Learning Objectives

- Compute Gaussian discriminant scores from estimated class statistics and priors.
- Visualise a Bayesian decision boundary in two dimensions.
- Explain how priors and misclassification costs move a decision threshold.
- Check probability and covariance assumptions before trusting predictions.

## Expected Output and Evidence

Retain the following evidence in your executed notebook:

- estimated Gaussian class statistics and posterior probabilities
- a labelled decision-boundary figure plus numeric held-out evidence
- a cost-sensitive decision comparison with stated assumptions

Do not include credentials, private data, or another learner's work.

## Prerequisites

- Conditional probability and Bayes' rule
- Means, covariance matrices, and matrix multiplication
- Basic plotting

## Session Plan

| Time | Activity | Evidence |
|---:|---|---|
| 0–10 min | Orientation and diagnostic prompt | Initial prediction or sketch |
| 10–30 min | Background concepts and setup | Concept notes and reproducible environment |
| 30–75 min | Guided practice | Executed analysis with interpretations |
| 75–105 min | Student challenge | Independent model choice and evidence |
| 105–115 min | Testing and debugging | Passing checks and resolved issues |
| 115–120 min | Submission and reflection | Concise evidence-based reflection |

## Background Concepts

Bayes' rule combines likelihood and prior:

$$P(C_k\mid x) \propto p(x\mid C_k)P(C_k).$$

With class means $\mu_k$, a shared covariance $\Sigma$, and priors $\pi_k$, the log discriminant is
$g_k(x)=x^T\Sigma^{-1}\mu_k-\frac12\mu_k^T\Sigma^{-1}\mu_k+\log\pi_k$.
Choose the class with the largest score. Equal covariance creates a linear boundary; unequal costs replace maximum posterior probability with minimum expected risk.

## Accessibility and Responsible Use

Every visual result in the guided path is paired with a printed numeric summary. Do not rely on colour alone in your challenge evidence: add labels, line styles, markers, or a compact table as appropriate. Add concise alt text when exporting figures for another format. Use only public, approved, or synthetic data.

Follow the collaboration, acknowledgement, and AI-use rules published by your current offering; this institution-neutral practical does not define those rules.

## Setup

Run the next cell before starting. All datasets are bundled with scikit-learn or generated locally; no download is required.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")

## Guided Practice

Follow the sequence and pause at each interpretation prompt before moving on.

### 1. Create and inspect two classes

We use generated data so the experiment is fully reproducible and the true class overlap is visible.

In [ ]:
X, y = make_blobs(
    n_samples=600, centers=[(-2.0, 0.0), (2.0, 1.0)],
    cluster_std=[1.35, 1.10], random_state=RANDOM_STATE,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", s=18, alpha=0.65)
plt.title("Training observations")
plt.show()

### 2. Estimate a shared-covariance Gaussian model

The pooled covariance encodes the assumption that both classes have the same feature covariance.

In [ ]:
classes = np.unique(y_train)
means = np.vstack([X_train[y_train == k].mean(axis=0) for k in classes])
centered = np.vstack([X_train[y_train == k] - means[k] for k in classes])
pooled_cov = centered.T @ centered / (len(X_train) - len(classes))
empirical_priors = np.array([(y_train == k).mean() for k in classes])
print("means:\n", means)
print("pooled covariance:\n", pooled_cov.round(3))
print("priors:", empirical_priors)

### 3. Compute posterior probabilities and predictions

Subtracting the row maximum before exponentiating keeps the calculation numerically stable.

In [ ]:
def discriminant_scores(X_values, means, covariance, priors):
    inverse = np.linalg.inv(covariance)
    scores = []
    for mean, prior in zip(means, priors):
        scores.append(X_values @ inverse @ mean - 0.5 * mean @ inverse @ mean + np.log(prior))
    return np.column_stack(scores)

def posterior_probabilities(X_values, means, covariance, priors):
    scores = discriminant_scores(X_values, means, covariance, priors)
    shifted = scores - scores.max(axis=1, keepdims=True)
    unnormalised = np.exp(shifted)
    return unnormalised / unnormalised.sum(axis=1, keepdims=True)

test_posteriors = posterior_probabilities(X_test, means, pooled_cov, empirical_priors)
test_predictions = classes[test_posteriors.argmax(axis=1)]
print(f"Held-out accuracy: {accuracy_score(y_test, test_predictions):.3f}")

### 4. Draw the decision boundary and change the priors

Compare the solid equal-prior boundary with the dashed boundary that favours class 1.

In [ ]:
x0 = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 220)
x1 = np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 220)
xx, yy = np.meshgrid(x0, x1)
grid = np.column_stack([xx.ravel(), yy.ravel()])

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm", s=22, alpha=0.65)
for priors, style, label in [
    (np.array([0.5, 0.5]), "-", "equal priors"),
    (np.array([0.2, 0.8]), "--", "class 1 prior = 0.8"),
]:
    difference = np.diff(discriminant_scores(grid, means, pooled_cov, priors), axis=1).reshape(xx.shape)
    ax.contour(xx, yy, difference, levels=[0], colors="black", linestyles=style)
ax.plot([], [], "k-", label="equal priors")
ax.plot([], [], "k--", label="class 1 prior = 0.8")
ax.legend()
ax.set_title("Bayesian decision boundaries")
plt.show()

## Student Challenge

Introduce asymmetric costs. Treat a false negative as five times as costly as a false positive. Convert the class-1 posterior to a minimum-risk prediction, compare its confusion counts with the equal-cost rule, and justify the threshold you used.

In [ ]:
def predict_with_costs(p_class1, cost_false_positive=1.0, cost_false_negative=1.0):
    # Choosing class 1 incurs FP cost when the true class is 0; choosing class 0 incurs FN cost.
    threshold = cost_false_positive / (cost_false_positive + cost_false_negative)
    return (p_class1 > threshold).astype(int)

# TODO: apply this function to `test_posteriors[:, 1]`, calculate confusion counts,
# and compare with `test_predictions`.
challenge_predictions = predict_with_costs(test_posteriors[:, 1], 1.0, 5.0)
challenge_predictions[:10]

### Challenge success criteria

- The cost-derived threshold is stated and justified.
- Both false-positive and false-negative counts are reported.
- The interpretation distinguishes probability estimation from the decision rule.
- The final recommendation names the assumed costs.

## Testing and Debugging

Run these checks after the guided practice. If one fails, inspect shapes, label order, random seeds, and whether preprocessing was fitted only on training data.

In [ ]:
# Exercise a failure mode: a singular covariance cannot be inverted.
try:
    discriminant_scores(X_test[:2], means, np.zeros_like(pooled_cov), empirical_priors)
except np.linalg.LinAlgError as error:
    print("Expected singular-covariance failure:", error)
else:
    raise AssertionError("A singular covariance should fail clearly.")

assert np.isfinite(X_test).all(), "Missing values require an explicit modelling decision."
print("Edge check: posterior rows are finite for the guided test data.")

In [ ]:
assert np.allclose(empirical_priors.sum(), 1.0)
assert np.all(np.linalg.eigvalsh(pooled_cov) > 0), "Covariance must be positive definite."
assert np.allclose(test_posteriors.sum(axis=1), 1.0)
assert set(np.unique(test_predictions)).issubset(set(classes))
print("M03A guided-practice checks passed.")

## Extension Task

Replace the shared covariance with a separate covariance for each class to obtain quadratic discriminants. Plot both boundaries and identify where the modelling assumption matters.

## Submission and Reflection

Submit your completed notebook (or an HTML export) with outputs visible. Include: 

1. the challenge code and result;
2. one figure or compact results table;
3. a 150–250 word reflection answering: **What changed, why did it change, and what evidence supports your explanation?**

Follow your offering's published collaboration, acknowledgement, and AI-use rules. You remain responsible for checking and explaining all submitted work.

## References

- Source: `F1A-01-DecisionBoundaries.ipynb`.
- Duda, Hart, and Stork, *Pattern Classification*, Chapter 2.
- scikit-learn User Guide: [Linear and Quadratic Discriminant Analysis](https://scikit-learn.org/stable/modules/lda_qda.html)